In [5]:
import csv
import pandas as pd
import ast
import lotus

# Common Utils

In [7]:
def read_csv(csv_file, index_col=-1):
    if index_col != -1:
        df = pd.read_csv(csv_file, index_col=index_col)
    else:
        df = pd.read_csv(csv_file)
    
    def parse_list_col(x):
        """
        Try to parse x (a string) as a Python list using literal_eval.
        If parsing fails, return x unchanged.
        """
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                # If x is not a valid Python literal, just return x
                return x
        return x
    
    # Convert string -> list for the specified columns
    for col in ["pred_reaction", "reactions_list", "_map"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list_col)
            
    return df

# Parse regular join output for rerank

In [3]:
# answers_df =  pd.read_csv("biodex_cascade_answers_for_lm_rerank.csv", index_col=0)
answers_df = read_csv("biodex_cascade_answers_for_lm_rerank.csv", index_col=0)

In [4]:
answers_df['reactions_list'].iloc[0]

['Heparin-induced thrombocytopenia']

In [5]:
def to_comma_separated(val):
    """
    Safely convert val (which could be a list or a string representing a list)
    into a comma-separated string.
    """
    if isinstance(val, list):
        # Already a list of strings
        return ", ".join(val)
    elif isinstance(val, str):
        # Possibly a string like "['foo', 'bar']"
        # Try literal_eval to see if it's a valid Python list
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                # It's a real list, convert to comma-separated
                return ", ".join(parsed)
            else:
                # Not a list—just return the original string
                return val
        except (SyntaxError, ValueError):
            # It's not a parseable list, return as is
            return val
    else:
        # Fallback: convert whatever it is to string
        return str(val)

# 2) Normalize reactions_list so every row is a comma-separated string
answers_df["reactions_list"] = answers_df["reactions_list"].apply(to_comma_separated)

# 3) Group by and aggregate
grouped_df = (
    answers_df
    .groupby(["title", "abstract", "reactions", "reactions_list", "patient_description"], dropna=False)
    .apply(lambda grp: grp["reaction"].tolist())
    .reset_index(name="pred_reaction")
)

# 4) Convert that comma-separated string (in grouped_df) back to a list
grouped_df["reactions_list"] = grouped_df["reactions_list"].apply(
    lambda s: s.split(", ")
)

/tmp/user/24031/ipykernel_3413155/2824786360.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp["reaction"].tolist())


In [6]:
grouped_df

,title,abstract,reactions,reactions_list,patient_description,pred_reaction
0,A Case Report of Acute Transient Encephalopath...,Methemoglobinemia is caused due to an increase...,"Facial paralysis, Methaemoglobinaemia","[Facial paralysis, Methaemoglobinaemia]",TITLE:\nA Case Report of Acute Transient Encep...,"[Cyanosis, Methaemoglobinaemia, Cyanosis centr..."
1,A Case Report of COVID-Associated Catastrophic...,Antiphospholipid syndrome (APS) is an autoimmu...,"Haemoglobin decreased, Off label use, Retroper...","[Haemoglobin decreased, Off label use, Retrope...",TITLE:\nA Case Report of COVID-Associated Cata...,"[Autoimmune haemolytic anaemia, Autoimmune ana..."
2,A Case of Hemorrhagic Cholecystitis and Hemobi...,BACKGROUND Hemorrhagic cholecystitis is a rare...,"Cholecystitis acute, Haemobilia, Haemorrhagic ...","[Cholecystitis acute, Haemobilia, Haemorrhagic...",TITLE:\nA Case of Hemorrhagic Cholecystitis an...,"[Haemorrhagic cholecystitis, Haemobilia, Cysti..."
3,A Case of Non-small Cell Lung Cancer Treated w...,The echinoderm microtubule associated protein-...,"Decreased appetite, Disease progression, Dyspn...","[Decreased appetite, Disease progression, Dysp...",TITLE:\nA Case of Non-small Cell Lung Cancer T...,"[Transaminases abnormal, Hypertransaminasaemia..."
4,A Case of Paraneoplastic Pemphigus as a Preced...,"Paraneoplastic pemphigus is a rare, life-threa...","Bronchopulmonary aspergillosis, Obliterative b...","[Bronchopulmonary aspergillosis, Obliterative ...",TITLE:\nA Case of Paraneoplastic Pemphigus as ...,"[Bronchopulmonary aspergillosis, Obliterative ..."
...,...,...,...,...,...,...
220,Unusual Presentation of Relapsing Polychondrit...,BACKGROUND Relapsing polychondritis (RP) is an...,"Blood glucose abnormal, Spinal compression fra...","[Blood glucose abnormal, Spinal compression fr...",TITLE:\nUnusual Presentation of Relapsing Poly...,"[Steroid diabetes, Steroid therapy, Transamina..."
221,Vertebral artery dissection in term pregnancy ...,BACKGROUND\nVertebral artery dissection is an ...,"Condition aggravated, Exposure during pregnanc...","[Condition aggravated, Exposure during pregnan...",TITLE:\nVertebral artery dissection in term pr...,"[Dizziness, Vomiting, Nausea]"
222,Vertical transmission: evidence of COVID-19 in...,This article reports the case of a 28-year-old...,"COVID-19, Drug ineffective, Exposure during pr...","[COVID-19, Drug ineffective, Exposure during p...",TITLE:\nVertical transmission: evidence of COV...,"[Lymphocytopenia neonatal, Tachycardia foetal,..."
223,Very severe aplastic anemia in an 80-year-old ...,Although the patient with very severe aplastic...,"Pneumonia, Product use in unapproved indicatio...","[Pneumonia, Product use in unapproved indicati...",TITLE:\nVery severe aplastic anemia in an 80-y...,"[Serum sickness, Sepsis, Neutropenic sepsis, P..."


In [8]:
top_k = 25
rerank_prompt = (
    f"Given {{patient_description}}, pick the {top_k} most applicable adverse drug reactions from the options "
    f"that are directly expressed in the following list: {{pred_reaction}}. "
    "Rank from most applicable to least applicable. "
    "Always write your answer as a list of comma-separated adverse drug reactions only and nothing else."
)

rerank_num_lm = grouped_df.shape[0]

grouped_df = grouped_df.sem_map(
    rerank_prompt
)
        

ValueError: The language model must be an instance of LM. Please configure a valid language model using lotus.settings.configure()

# Parse rerank output

In [71]:
# top_answer_df = pd.read_csv("biodex_reranked_answers.csv", index_col=0)
top_answer_df = read_csv("biodex_reranked_answers.csv", index_col=0)


In [72]:
top_answer_df

,title,abstract,reactions,reactions_list,patient_description,pred_reaction,_map
0,A Case Report of Acute Transient Encephalopath...,Methemoglobinemia is caused due to an increase...,"Facial paralysis, Methaemoglobinaemia","[Facial paralysis, Methaemoglobinaemia]",TITLE:\nA Case Report of Acute Transient Encep...,"[Cyanosis, Methaemoglobinaemia, Cyanosis centr...","Cyanosis, Methaemoglobinaemia, Hypoxia, Cyanos..."
1,A Case Report of COVID-Associated Catastrophic...,Antiphospholipid syndrome (APS) is an autoimmu...,"Haemoglobin decreased, Off label use, Retroper...","[Haemoglobin decreased, Off label use, Retrope...",TITLE:\nA Case Report of COVID-Associated Cata...,"[Autoimmune haemolytic anaemia, Autoimmune ana...","Thrombocytopenia, Haemolytic anaemia, Autoimmu..."
2,A Case of Hemorrhagic Cholecystitis and Hemobi...,BACKGROUND Hemorrhagic cholecystitis is a rare...,"Cholecystitis acute, Haemobilia, Haemorrhagic ...","[Cholecystitis acute, Haemobilia, Haemorrhagic...",TITLE:\nA Case of Hemorrhagic Cholecystitis an...,"[Haemorrhagic cholecystitis, Haemobilia, Cysti...","Haemorrhagic cholecystitis, Haemobilia, Haemor..."
3,A Case of Non-small Cell Lung Cancer Treated w...,The echinoderm microtubule associated protein-...,"Decreased appetite, Disease progression, Dyspn...","[Decreased appetite, Disease progression, Dysp...",TITLE:\nA Case of Non-small Cell Lung Cancer T...,"[Transaminases abnormal, Hypertransaminasaemia...","Transaminases abnormal, Hypertransaminasaemia,..."
4,A Case of Paraneoplastic Pemphigus as a Preced...,"Paraneoplastic pemphigus is a rare, life-threa...","Bronchopulmonary aspergillosis, Obliterative b...","[Bronchopulmonary aspergillosis, Obliterative ...",TITLE:\nA Case of Paraneoplastic Pemphigus as ...,"[Bronchopulmonary aspergillosis, Obliterative ...",Here is the list of 25 most applicable adverse...
...,...,...,...,...,...,...,...
220,Unusual Presentation of Relapsing Polychondrit...,BACKGROUND Relapsing polychondritis (RP) is an...,"Blood glucose abnormal, Spinal compression fra...","[Blood glucose abnormal, Spinal compression fr...",TITLE:\nUnusual Presentation of Relapsing Poly...,"[Steroid diabetes, Steroid therapy, Transamina...","Steroid diabetes, Steroid therapy, Transaminas..."
221,Vertebral artery dissection in term pregnancy ...,BACKGROUND\nVertebral artery dissection is an ...,"Condition aggravated, Exposure during pregnanc...","[Condition aggravated, Exposure during pregnan...",TITLE:\nVertebral artery dissection in term pr...,"[Dizziness, Vomiting, Nausea]","Dizziness, Vomiting, Nausea"
222,Vertical transmission: evidence of COVID-19 in...,This article reports the case of a 28-year-old...,"COVID-19, Drug ineffective, Exposure during pr...","[COVID-19, Drug ineffective, Exposure during p...",TITLE:\nVertical transmission: evidence of COV...,"[Lymphocytopenia neonatal, Tachycardia foetal,...","Foetal tachycardia, Tachycardia foetal, Neonat..."
223,Very severe aplastic anemia in an 80-year-old ...,Although the patient with very severe aplastic...,"Pneumonia, Product use in unapproved indicatio...","[Pneumonia, Product use in unapproved indicati...",TITLE:\nVery severe aplastic anemia in an 80-y...,"[Serum sickness, Sepsis, Neutropenic sepsis, P...",Here is the list of 25 most applicable adverse...


In [83]:
top_k = 25
known_prefixes = [
    f"Based on the patient description, the {top_k} most applicable adverse drug reactions are:\n\n",
    f"Based on the Patient_description, the {top_k} most applicable adverse drug reactions from the Combined_reaction_list are:\n\n",
    f"Based on the Patient_description, the {top_k} most applicable adverse drug reactions are:\n\n",
    f"Based on the provided Patient_description, the {top_k} most applicable adverse drug reactions from the Combined_reaction_list are:\n\n",
    f"Here is the list of {top_k} most applicable adverse drug reactions:\n\n",
    "Here is the answer:\n\n",
    f"Here is the list of the {top_k} most applicable adverse drug reactions:\n\n",
    f"Here is the list of {top_k} most applicable adverse drug reactions:\n\n",
    f"Here is the list of {top_k} most applicable adverse drug reactions from the options, ranked from most applicable to least applicable:"
]
def remove_known_prefixes(text: str, prefixes: list) -> str:
    """
    Removes the first matching prefix from 'text' if found in 'prefixes',
    otherwise returns text unchanged.
    """
    for prefix in prefixes:
        if text.startswith(prefix):
            return text[len(prefix):]
    return text

In [84]:
top_answer_df["_map"] = top_answer_df["_map"].fillna("").apply(
    lambda x: remove_known_prefixes(x, known_prefixes)
)


In [85]:
for i, row in top_answer_df.iterrows():
    # print(f"Question {i}:")
    print(row["_map"])

Cyanosis, Methaemoglobinaemia, Hypoxia, Cyanosis central, Cyanopsia, Dysglobulinaemia, Congenital methaemoglobinaemia, Hypochromasia
Thrombocytopenia, Haemolytic anaemia, Autoimmune haemolytic anaemia, Autoimmune anaemia, Thrombocytopenia, Platelet dysfunction, Renal impairment, Renal failure, Autoimmune pancytopenia, Renal haemorrhage, Thrombocytopenia, Haemolytic anaemia, Autoimmune haemolytic anaemia, Autoimmune anaemia, Platelet dysfunction, Renal impairment, Renal failure, Autoimmune pancytopenia, Renal haemorrhage, Thrombocytopenia, Haemolytic anaemia, Autoimmune haemolytic anaemia, Autoimmune anaemia, Platelet dysfunction, Renal impairment.
Haemorrhagic cholecystitis, Haemobilia, Haemorrhage, Gastrointestinal haemorrhage, Upper gastrointestinal haemorrhage, Gastric haemorrhage, Gastric ulcer haemorrhage, Haemorrhagic gastritis, Haemorrhagic cyst, Cholecystitis, Enterocolitis haemorrhagic, Rectal haemorrhage, Diarrhoea haemorrhagic, Haemorrhagic erosive gastritis, Gastrointestina

In [86]:
top_answer_df.rename(columns={"pred_reaction": "pred_reaction_norank"}, inplace=True)
top_answer_df["pred_reaction"] = top_answer_df["_map"].apply(
    lambda x: [reaction.strip() for reaction in x.split(",") if reaction.strip()]
)
# top_answer_df["pred_reaction"] = top_answer_df["pred_reaction"].apply(
#     lambda reactions: [f"'{r}'" for r in reactions]
# )

In [87]:
top_answer_df['pred_reaction'].iloc[0]

['Cyanosis',
 'Methaemoglobinaemia',
 'Hypoxia',
 'Cyanosis central',
 'Cyanopsia',
 'Dysglobulinaemia',
 'Congenital methaemoglobinaemia',
 'Hypochromasia']

In [88]:
from biodex.metrics import compute_precision, compute_rank_precision, compute_recall

def compute_metrics(res_df, gt_col_name="reactions_list", pred_col_name="pred_reaction") -> pd.DataFrame:
    res_df["rank_precision@5"] = res_df.apply(
        lambda x: compute_rank_precision(x[gt_col_name], x[pred_col_name], cutoff=5),
        axis=1,
    )
    res_df["rank-precision@10"] = res_df.apply(
        lambda x: compute_rank_precision(x[gt_col_name], x[pred_col_name], cutoff=10),
        axis=1,
    )
    res_df["rank-precision@25"] = res_df.apply(
        lambda x: compute_rank_precision(x[gt_col_name], x[pred_col_name], cutoff=25),
        axis=1,
    )


    for k in [5, 10, 20, 50, 100, 200, 300, 400, 500]:
        res_df[f"recall@{k}"] = res_df.apply(lambda x: compute_recall(x[gt_col_name], x[pred_col_name], k), axis=1)

    res_df["precision@5"] = res_df.apply(lambda x: compute_precision(x[gt_col_name], x[pred_col_name], 5), axis=1)
    res_df["precision@10"] = res_df.apply(lambda x: compute_precision(x[gt_col_name], x[pred_col_name], 10), axis=1)

    res_df["num_ids"] = res_df.apply(lambda x: len(x[pred_col_name]), axis=1)

    # take subset of df with metrics
    df = res_df[[col for col in res_df.columns if "@" in col or "latency" in col or "num_ids" in col]]

    return df

In [89]:

metrics_df = compute_metrics(top_answer_df, gt_col_name="reactions_list", pred_col_name="pred_reaction")

gt_ids: ['Facial paralysis', 'Methaemoglobinaemia'], type: <class 'list'>
ids: ['Cyanosis', 'Methaemoglobinaemia', 'Hypoxia', 'Cyanosis central', 'Cyanopsia', 'Dysglobulinaemia', 'Congenital methaemoglobinaemia', 'Hypochromasia'], type: <class 'list'>
gt_ids: ['Haemoglobin decreased', 'Off label use', 'Retroperitoneal haemorrhage'], type: <class 'list'>
ids: ['Thrombocytopenia', 'Haemolytic anaemia', 'Autoimmune haemolytic anaemia', 'Autoimmune anaemia', 'Thrombocytopenia', 'Platelet dysfunction', 'Renal impairment', 'Renal failure', 'Autoimmune pancytopenia', 'Renal haemorrhage', 'Thrombocytopenia', 'Haemolytic anaemia', 'Autoimmune haemolytic anaemia', 'Autoimmune anaemia', 'Platelet dysfunction', 'Renal impairment', 'Renal failure', 'Autoimmune pancytopenia', 'Renal haemorrhage', 'Thrombocytopenia', 'Haemolytic anaemia', 'Autoimmune haemolytic anaemia', 'Autoimmune anaemia', 'Platelet dysfunction', 'Renal impairment.'], type: <class 'list'>
gt_ids: ['Cholecystitis acute', 'Haemobili

In [91]:
metrics_df.mean()

rank_precision@5      0.302519
rank-precision@10     0.335506
rank-precision@25     0.476046
recall@5              0.243553
recall@10             0.266467
recall@20             0.293134
recall@50             0.295293
recall@100            0.295293
recall@200            0.295293
recall@300            0.295293
recall@400            0.295293
recall@500            0.295293
precision@5           0.174593
precision@10          0.115601
num_ids              19.435556
dtype: float64

In [81]:
norank_metrics = compute_metrics(top_answer_df, gt_col_name="reactions_list", pred_col_name="pred_reaction_norank")

gt_ids: ['Facial paralysis', 'Methaemoglobinaemia'], type: <class 'list'>
ids: ['Cyanosis', 'Methaemoglobinaemia', 'Cyanosis central', 'Cyanopsia', 'Dysglobulinaemia', 'Congenital methaemoglobinaemia', 'Hypoxia', 'Hypochromasia'], type: <class 'list'>
gt_ids: ['Haemoglobin decreased', 'Off label use', 'Retroperitoneal haemorrhage'], type: <class 'list'>
ids: ['Autoimmune haemolytic anaemia', 'Autoimmune anaemia', 'Renal failure', 'Autoimmune pancytopenia', 'Thrombocytopenia', 'Haemolytic anaemia', 'Renal haemorrhage', 'Platelet dysfunction', 'Renal impairment'], type: <class 'list'>
gt_ids: ['Cholecystitis acute', 'Haemobilia', 'Haemorrhagic ascites', 'Haemorrhagic cholecystitis'], type: <class 'list'>
ids: ['Haemorrhagic cholecystitis', 'Haemobilia', 'Cystitis haemorrhagic', 'Hepatic haemorrhage', 'Gastrointestinal haemorrhage', 'Haemorrhage', 'Haemorrhagic disorder', 'Intestinal varices haemorrhage', 'Haemorrhoidal haemorrhage', 'Intestinal haemorrhage', 'Haemorrhagic hepatic cyst', 

In [82]:
norank_metrics.mean()

rank_precision@5      0.239185
rank-precision@10     0.251441
rank-precision@25     0.273030
recall@5              0.212867
recall@10             0.246842
recall@20             0.271991
recall@50             0.284028
recall@100            0.287659
recall@200            0.293215
recall@300            0.293215
recall@400            0.293215
recall@500            0.293215
precision@5           0.157778
precision@10          0.113062
num_ids              25.346667
dtype: float64

# Check the normal output

In [3]:

ref_df = pd.read_csv("biodex_results/JoinCascade/nsamples=250_recall_target=0.7_precision_target=0.7_range=150/res.csv")

In [4]:
ref_df.head(5)

,title,abstract,reactions,reactions_list,pred_reaction
0,A Case Report of Acute Transient Encephalopath...,Methemoglobinemia is caused due to an increase...,"Facial paralysis, Methaemoglobinaemia","['Facial paralysis', 'Methaemoglobinaemia']","['Cyanosis', 'Methaemoglobinaemia', 'Cyanosis ..."
1,A Case Report of COVID-Associated Catastrophic...,Antiphospholipid syndrome (APS) is an autoimmu...,"Haemoglobin decreased, Off label use, Retroper...","['Haemoglobin decreased', 'Off label use', 'Re...","['Autoimmune haemolytic anaemia', 'Autoimmune ..."
2,A Case of Hemorrhagic Cholecystitis and Hemobi...,BACKGROUND Hemorrhagic cholecystitis is a rare...,"Cholecystitis acute, Haemobilia, Haemorrhagic ...","['Cholecystitis acute', 'Haemobilia', 'Haemorr...","['Haemorrhagic cholecystitis', 'Haemobilia', '..."
3,A Case of Non-small Cell Lung Cancer Treated w...,The echinoderm microtubule associated protein-...,"Decreased appetite, Disease progression, Dyspn...","['Decreased appetite', 'Disease progression', ...","['Transaminases abnormal', 'Hypertransaminasae..."
4,A Case of Paraneoplastic Pemphigus as a Preced...,"Paraneoplastic pemphigus is a rare, life-threa...","Bronchopulmonary aspergillosis, Obliterative b...","['Bronchopulmonary aspergillosis', 'Obliterati...","['Bronchopulmonary aspergillosis', 'Obliterati..."


In [5]:
ref_metrics = compute_metrics(ref_df)
ref_metrics.mean()

gt_ids: ['Facial paralysis', 'Methaemoglobinaemia'], type: <class 'str'>
ids: ['Cyanosis', 'Methaemoglobinaemia', 'Cyanosis central', 'Cyanopsia', 'Dysglobulinaemia', 'Congenital methaemoglobinaemia', 'Hypoxia', 'Hypochromasia'], type: <class 'str'>
gt_ids: ['Haemoglobin decreased', 'Off label use', 'Retroperitoneal haemorrhage'], type: <class 'str'>
ids: ['Autoimmune haemolytic anaemia', 'Autoimmune anaemia', 'Renal failure', 'Autoimmune pancytopenia', 'Thrombocytopenia', 'Haemolytic anaemia', 'Renal haemorrhage', 'Platelet dysfunction', 'Renal impairment'], type: <class 'str'>
gt_ids: ['Cholecystitis acute', 'Haemobilia', 'Haemorrhagic ascites', 'Haemorrhagic cholecystitis'], type: <class 'str'>
ids: ['Haemorrhagic cholecystitis', 'Haemobilia', 'Cystitis haemorrhagic', 'Hepatic haemorrhage', 'Gastrointestinal haemorrhage', 'Haemorrhage', 'Haemorrhagic disorder', 'Intestinal varices haemorrhage', 'Haemorrhoidal haemorrhage', 'Intestinal haemorrhage', 'Haemorrhagic hepatic cyst', 'Uppe

rank_precision@5       0.860714
rank-precision@10      0.876091
recall@5               0.078515
recall@10              0.142978
recall@20              0.220267
recall@50              0.278794
recall@100             0.304818
recall@200             0.321218
recall@300             0.329600
recall@400             0.334181
recall@500             0.337069
precision@5            0.855357
precision@10           0.798661
num_ids              657.575893
dtype: float64

In [8]:
ref_metrics

,rank_precision@5,rank-precision@10,recall@5,recall@10,recall@20,recall@50,recall@100,recall@200,recall@300,recall@400,recall@500,precision@5,precision@10,num_ids
0,0.8,0.9,0.093023,0.186047,0.325581,0.441860,0.488372,0.511628,0.511628,0.511628,0.511628,0.8,0.8,150
1,0.8,0.9,0.054795,0.095890,0.178082,0.205479,0.260274,0.328767,0.342466,0.342466,0.342466,0.8,0.7,206
2,1.0,1.0,0.054945,0.098901,0.153846,0.219780,0.219780,0.219780,0.219780,0.219780,0.230769,1.0,0.9,3741
3,0.8,0.8,0.058824,0.102941,0.147059,0.205882,0.250000,0.250000,0.250000,0.250000,0.250000,0.8,0.7,89
4,1.0,1.0,0.048077,0.086538,0.144231,0.230769,0.240385,0.240385,0.240385,0.250000,0.250000,1.0,0.9,2457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,1.0,1.0,0.087719,0.175439,0.245614,0.298246,0.333333,0.350877,0.403509,0.403509,0.403509,1.0,1.0,655
220,0.6,0.7,0.044776,0.089552,0.149254,0.208955,0.208955,0.208955,0.208955,0.208955,0.208955,0.6,0.6,35
221,1.0,1.0,0.012225,0.022005,0.036675,0.051345,0.053790,0.061125,0.066015,0.066015,0.073350,1.0,0.9,447
222,1.0,1.0,0.079365,0.158730,0.206349,0.253968,0.301587,0.301587,0.301587,0.301587,0.317460,1.0,1.0,950


# Map df

In [4]:
map_df = read_csv("map_output.csv")

UnboundLocalError: local variable 'df' referenced before assignment